In [1]:
import os


os.environ["HF_HOME"] = "/vol/bitbucket/m24/.cache"
os.environ["MODELSCOPE_CACHE"] = os.path.join(os.environ["HF_HOME"], "modelscope")
os.environ["DIFFUSERS_CACHE"] = os.path.join(os.environ["HF_HOME"], "diffusers")
os.environ["HF_DATASETS_CACHE"] = os.path.join(os.environ["HF_HOME"], "datasets")
os.environ["MPLCONFIGDIR"] = "/vol/bitbucket/m24/.cache/matplotlib"

# Contributing a New Metric to eval-unlearn

This notebook is a step-by-step guide for contributors who want to add a new
evaluation metric to the **eval-unlearn** library.

It covers:
1. The required file structure
2. How to write a `Config` dataclass
3. How to implement the three-method metric interface (`__init__`, `update`, `compute`)
4. How to register the metric as a plugin via `pyproject.toml`
5. **Validation tests** — the full suite your metric must pass before a PR is merged

Each test cell is self-contained and prints a clear pass/fail verdict. Run all
cells top-to-bottom to verify your implementation.

> **CPU is sufficient** for all structural/unit tests. Heavy model downloads are
> mocked throughout.

## Prerequisites

```bash
# Install eval-unlearn in editable mode so your new package is importable
pip install -e /path/to/eval-unlearn-testing/Packages/eval-unlearn
pip install pytest
```

Any extra packages your metric depends on (e.g. a detection model) must also be
installed.

In [2]:
# # ── Set this to your new metric's registered name ────────────────────────────
# METRIC_NAME   = "my_metric"          # e.g. "my_metric"

# # ── Module path (dotted) to your metric class ─────────────────────────────────
# METRIC_MODULE = f"eval_unlearn.metrics.{METRIC_NAME}.metric"   # adjust if different
# METRIC_CLASS  = "MyMetric"           # the class name inside metric.py

# # ── Config class name (convention is {METRIC_CLASS}Config; override if different) ──
# CONFIG_CLASS  = None   # set to e.g. "MyConfig" only if you deviated from convention

# # ── Metric scoring pattern ──────────────────────────────────────────────────
# # "per_image"    → metric scores each image individually (standard template)
# # "distribution" → metric compares full distributions (e.g. FID); no per-image scores
# METRIC_PATTERN = "per_image"

# # ── Minimal valid config kwargs (must construct with no errors) ───────────────
# REQUIRED_CONFIG = {
#     "device": "cpu",
# }

# # ── External package your metric imports (if any; else set to None) ──────────
# EXTERNAL_PACKAGE = None   # e.g. "nudenet", or None if no external package

# print(f"Metric name  : {METRIC_NAME}")
# print(f"Module path  : {METRIC_MODULE}")
# print(f"External pkg : {EXTERNAL_PACKAGE}")

#Testing
# ── Set this to your new metric's registered name ────────────────────────────
METRIC_NAME   = "fid"                # from @register_metric("fid") in metric.py

# ── Module path (dotted) to your metric class ─────────────────────────────────
METRIC_MODULE = "eval_unlearn.metrics.fid.metric"   # src/eval_unlearn/metrics/fid/metric.py
METRIC_CLASS  = "FIDMetric"          # class FIDMetric: at line 57

# ── Config class name (convention is {METRIC_CLASS}Config; override if your class differs) ──
CONFIG_CLASS  = "FIDConfig"   # the Config dataclass name inside config.py

# ── Metric scoring pattern ──────────────────────────────────────────────────
# "per_image"    → metric scores each image individually (standard template)
# "distribution" → metric compares full distributions (e.g. FID); no per-image scores
METRIC_PATTERN = "distribution"

# ── Minimal valid config kwargs (must construct with no errors) ───────────────
REQUIRED_CONFIG = {
    "device": "cpu",
}

# ── External package your metric imports (if any; else set to None) ──────────
EXTERNAL_PACKAGE = None   # only uses numpy, torch, torchvision, scipy, PIL — all standard


---
## Part 1 — File Structure

Every metric lives in its own sub-package inside `src/eval_unlearn/metrics/`:

```
src/eval_unlearn/metrics/
└── my_metric/
    ├── __init__.py      # (empty)
    ├── config.py        # Config dataclass
    └── metric.py        # Metric class with __init__, update, compute
```

### `config.py` template

```python
from dataclasses import dataclass
from typing import Optional
from ...configs.base import BaseConfig

@dataclass(frozen=True)
class MyMetricConfig(BaseConfig):
    device: Optional[str] = None      # None → auto-detect
    limit: Optional[int] = 300        # max prompts to stream from dataset
    # Add any metric-specific parameters here
    threshold: float = 0.5

    def __post_init__(self):
        if self.threshold <= 0 or self.threshold >= 1:
            raise ValueError("threshold must be in (0, 1)")

    @classmethod
    def from_dict(cls, data):
        return super().from_dict(data)
```

### `metric.py` template

```python
from typing import Any, Dict, List, Optional
from torch.utils.data import DataLoader
from ...types import MetricResult
from ...registry import register_metric
from ...logging_utils import get_logger
from .config import MyMetricConfig

logger = get_logger(__name__)

try:
    import torch
    # import your detection/scoring model here
except ImportError as e:
    raise ImportError(
        "MyMetric requires 'torch'. Install with: pip install eval-unlearn"
    ) from e

@register_metric("my_metric")
class MyMetric:
    def __init__(self, **kwargs):
        self.config = MyMetricConfig.from_dict(kwargs)
        self.device = self.config.device or (
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        # load your model here
        self._total_score = 0.0
        self._evaluated_count = 0
        self._total_count = 0
        self._per_image_scores = []

    def load_dataset(self) -> DataLoader:
        """Return a DataLoader of (prompts, metadata) tuples for this metric."""
        # Reset accumulators so the metric can be reused
        self._total_score = 0.0
        self._evaluated_count = 0
        self._total_count = 0
        self._per_image_scores = []
        # Return your DataLoader here
        raise NotImplementedError

    def update(
        self,
        images: List[Any],
        prompts: List[str],
        _metadata: Optional[Dict[str, Any]] = None,
    ) -> None:
        """Score each image-prompt pair and accumulate. Do NOT store raw images."""
        for img, prompt in zip(images, prompts):
            score = self._score_one(img, prompt)
            self._per_image_scores.append(score)
            if score is not None:
                self._total_score += score
                self._evaluated_count += 1
            self._total_count += 1

    def _score_one(self, img, prompt) -> Optional[float]:
        """Return a float in [0, 1] for this image, or None if scoring failed."""
        raise NotImplementedError

    def compute(self) -> MetricResult:
        """Average accumulated scores and return a MetricResult."""
        if self._total_count == 0:
            return MetricResult(name="MyMetric", value=0.0,
                                details={"error": "No images evaluated"})
        avg = self._total_score / self._evaluated_count if self._evaluated_count else 0.0
        return MetricResult(
            name="MyMetric",
            value=avg,
            details={
                "per_image_scores": self._per_image_scores,
                "evaluated_count": self._evaluated_count,
                "total_count": self._total_count,
                "config": self.config.to_dict(),
            },
        )
```

---
## Part 2 — Plugin Registration

Add one entry to `pyproject.toml`:

```toml
[project.entry-points."eval_unlearn.metrics"]
my_metric = "eval_unlearn.metrics.my_metric.metric:MyMetric"
```

Optionally, document the model your metric uses in
`src/eval_unlearn/metrics/_base_models.py`:

```python
METRIC_MODELS = {
    ...
    "my_metric": MetricModelInfo("my-model/name", configurable=False),
}
```

After editing `pyproject.toml` re-install the package:
```bash
pip install -e .
```

---
## Part 3 — Validation Tests

The cells below are the **exact checks** the maintainers run against every new
metric. Each cell prints `PASS` or `FAIL` with a reason.

All tests must show `PASS` before you open a pull request.

In [3]:
# ── Shared helpers ────────────────────────────────────────────────────────────
import sys
import importlib
import traceback
from unittest.mock import MagicMock, patch, Mock
from PIL import Image

def _dummy_image(w=64, h=64):
    return Image.new("RGB", (w, h), color=(100, 149, 237))

def _reload_metric_module():
    """Drop and reload the metric module so patches take effect cleanly."""
    sys.modules.pop(METRIC_MODULE, None)
    return importlib.import_module(METRIC_MODULE)

def _pass(label):
    print(f"  ✓  PASS  {label}")

def _fail(label, reason):
    print(f"  ✗  FAIL  {label}")
    print(f"           Reason: {reason}")

def _run(label, fn):
    try:
        fn()
        _pass(label)
    except AssertionError as e:
        _fail(label, str(e))
    except Exception:
        _fail(label, traceback.format_exc(limit=3))

def _patch_external():
    """Return a context manager that mocks the external package (if any)."""
    if EXTERNAL_PACKAGE:
        return patch.dict("sys.modules", {EXTERNAL_PACKAGE: MagicMock()})
    else:
        from contextlib import nullcontext
        return nullcontext()

print("Helpers loaded.")

Helpers loaded.


### Test Group 1 — Config Dataclass

The `Config` must:
- Be importable as `eval_unlearn.metrics.<name>.config`
- Accept all required kwargs without error
- Reject invalid values with `ValueError`
- Round-trip through `from_dict` / `to_dict`
- Inherit from `BaseConfig` and be frozen

In [4]:
print("=" * 60)
print("Test Group 1: Config Dataclass")
print("=" * 60)

CONFIG_MODULE = f"eval_unlearn.metrics.{METRIC_NAME}.config"
# Use CONFIG_CLASS from setup cell if defined; otherwise fall back to naming convention
CONFIG_CLASS = globals().get("CONFIG_CLASS") or f"{METRIC_CLASS}Config"

from eval_unlearn.configs.base import BaseConfig

# --------------------------------------------------------------------------
# T1.1  Config module is importable
# --------------------------------------------------------------------------
def t1_1_config_importable():
    mod = importlib.import_module(CONFIG_MODULE)
    assert hasattr(mod, CONFIG_CLASS), (
        f"{CONFIG_CLASS} not found in {CONFIG_MODULE}. "
        f"Available names: {[x for x in dir(mod) if not x.startswith('_')]}"
    )

_run("T1.1  Config module is importable", t1_1_config_importable)

try:
    _config_mod = importlib.import_module(CONFIG_MODULE)
    ConfigCls   = getattr(_config_mod, CONFIG_CLASS)
except Exception as e:
    print(f"  [SKIP remaining config tests — cannot import config: {e}]")
    ConfigCls = None

# --------------------------------------------------------------------------
# T1.2  Instantiation with required kwargs succeeds
# --------------------------------------------------------------------------
def t1_2_instantiation():
    assert ConfigCls is not None
    cfg = ConfigCls(**REQUIRED_CONFIG)
    assert cfg is not None

_run("T1.2  Instantiation with required kwargs succeeds", t1_2_instantiation)

# --------------------------------------------------------------------------
# T1.3  Inherits from BaseConfig
# --------------------------------------------------------------------------
def t1_3_inherits_base_config():
    assert ConfigCls is not None
    assert issubclass(ConfigCls, BaseConfig), (
        f"{CONFIG_CLASS} does not inherit from BaseConfig. "
        "Change: class {CONFIG_CLASS}(BaseConfig):"
    )

_run("T1.3  Inherits from BaseConfig", t1_3_inherits_base_config)

# --------------------------------------------------------------------------
# T1.4  Config is a frozen dataclass
# --------------------------------------------------------------------------
def t1_4_frozen():
    assert ConfigCls is not None
    cfg = ConfigCls(**REQUIRED_CONFIG)
    try:
        cfg.device = "changed"
        raise AssertionError(
            "Config must be frozen. Add frozen=True to @dataclass."
        )
    except Exception as e:
        if "cannot assign" in str(e).lower() or "frozen" in str(e).lower() or "FrozenInstanceError" in type(e).__name__:
            pass
        else:
            raise

_run("T1.4  Config is frozen (immutable)", t1_4_frozen)

# --------------------------------------------------------------------------
# T1.5  from_dict round-trips correctly
# --------------------------------------------------------------------------
def t1_5_from_dict_roundtrip():
    assert ConfigCls is not None
    cfg1 = ConfigCls(**REQUIRED_CONFIG)
    cfg2 = ConfigCls.from_dict(REQUIRED_CONFIG)
    assert cfg1.device == cfg2.device, (
        f"from_dict mismatch on 'device': {cfg1.device!r} vs {cfg2.device!r}"
    )

_run("T1.5  from_dict round-trips correctly", t1_5_from_dict_roundtrip)

# --------------------------------------------------------------------------
# T1.6  to_dict returns a dict containing 'device'
# --------------------------------------------------------------------------
def t1_6_to_dict():
    assert ConfigCls is not None
    cfg = ConfigCls(**REQUIRED_CONFIG)
    d   = cfg.to_dict()
    assert isinstance(d, dict), f"to_dict() must return dict, got {type(d)}"
    assert "device" in d, f"to_dict() result must contain 'device'. Keys: {list(d.keys())}"

_run("T1.6  to_dict returns dict containing 'device'", t1_6_to_dict)

# --------------------------------------------------------------------------
# T1.7  from_dict ignores unknown keys
# --------------------------------------------------------------------------
def t1_7_from_dict_ignores_extras():
    assert ConfigCls is not None
    extra_kwargs = {**REQUIRED_CONFIG, "__nonexistent_key__": 999}
    cfg = ConfigCls.from_dict(extra_kwargs)   # must not raise
    assert cfg is not None

_run("T1.7  from_dict ignores unknown keys", t1_7_from_dict_ignores_extras)


Test Group 1: Config Dataclass


/vol/bitbucket/m24/eval-unlearn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ✓  PASS  T1.1  Config module is importable
  ✓  PASS  T1.2  Instantiation with required kwargs succeeds
  ✓  PASS  T1.3  Inherits from BaseConfig
  ✓  PASS  T1.4  Config is frozen (immutable)
  ✓  PASS  T1.5  from_dict round-trips correctly
  ✓  PASS  T1.6  to_dict returns dict containing 'device'
  ✓  PASS  T1.7  from_dict ignores unknown keys


### Test Group 2 — Metric Class Interface

The `Metric` class must:
- Be importable and decorated with `@register_metric`
- Initialise with the required kwargs
- Set `self.device` and accumulator attributes (`_total_score`, `_evaluated_count`, `_total_count`, `_per_image_scores`)
- Auto-detect the device when `device=None`

In [5]:
print("=" * 60)
print("Test Group 2: Metric Class — Initialisation")
print("=" * 60)

def _make_metric(**extra_kwargs):
    """Instantiate the metric with any heavy model loading mocked out."""
    kwargs = {**REQUIRED_CONFIG, **extra_kwargs}
    with _patch_external():
        mod = _reload_metric_module()
        cls = getattr(mod, METRIC_CLASS)
        # Patch any torch/transformers imports the metric might do at __init__ time
        with patch(f"{METRIC_MODULE}.torch") as mock_torch:
            mock_torch.cuda.is_available.return_value = False
            try:
                metric = cls(**kwargs)
            except Exception:
                # Fallback: try without mocking torch (metric may not import it at module level)
                metric = cls(**kwargs)
    return metric

# --------------------------------------------------------------------------
# T2.1  Metric module is importable
# --------------------------------------------------------------------------
def t2_1_metric_importable():
    with _patch_external():
        mod = _reload_metric_module()
    assert hasattr(mod, METRIC_CLASS), (
        f"{METRIC_CLASS} not found in {METRIC_MODULE}."
    )

_run("T2.1  Metric module is importable", t2_1_metric_importable)

# --------------------------------------------------------------------------
# T2.2  Metric initialises with required kwargs
# --------------------------------------------------------------------------
def t2_2_init_succeeds():
    metric = _make_metric()
    assert metric is not None

_run("T2.2  Metric __init__ succeeds with required kwargs", t2_2_init_succeeds)

# --------------------------------------------------------------------------
# T2.3  self.device is set after init
# --------------------------------------------------------------------------
def t2_3_device_attribute():
    metric = _make_metric(device="cpu")
    assert hasattr(metric, "device"), (
        "Metric must set self.device in __init__."
    )
    assert metric.device == "cpu", (
        f"Expected device='cpu', got {metric.device!r}"
    )

_run("T2.3  self.device is set correctly", t2_3_device_attribute)

# --------------------------------------------------------------------------
# T2.4  Accumulator attributes are initialised to zero / empty
#
# per_image    → checks the four standard score accumulators
# distribution → checks the generated-feature-list accumulator
# --------------------------------------------------------------------------
def t2_4_accumulators_initialised():
    metric = _make_metric()
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        checks = [
            ("_total_score",       0.0,  float),
            ("_evaluated_count",   0,    int),
            ("_total_count",       0,    int),
            ("_per_image_scores",  [],   list),
        ]
        for attr, expected_val, expected_type in checks:
            assert hasattr(metric, attr), (
                f"Metric is missing accumulator attribute '{attr}'. "
                "The runner relies on these names."
            )
            val = getattr(metric, attr)
            assert isinstance(val, expected_type), (
                f"{attr} must be {expected_type.__name__}, got {type(val).__name__}"
            )
            assert val == expected_val, (
                f"{attr} must start at {expected_val!r}, got {val!r}"
            )
    else:
        # Distribution metrics accumulate feature arrays rather than per-image scores.
        # They must start with an empty generated-features list.
        assert hasattr(metric, "_gen_activations"), (
            "Distribution metric must initialise '_gen_activations' (list) in __init__."
        )
        assert isinstance(metric._gen_activations, list), (
            f"_gen_activations must be a list, got {type(metric._gen_activations).__name__}"
        )
        assert len(metric._gen_activations) == 0, (
            f"_gen_activations must start empty, got {len(metric._gen_activations)} items."
        )

_run("T2.4  Accumulator attributes initialised to zero/empty", t2_4_accumulators_initialised)


Test Group 2: Metric Class — Initialisation
  ✓  PASS  T2.1  Metric module is importable
2026-05-27 16:22:22 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:23 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
  ✓  PASS  T2.2  Metric __init__ succeeds with required kwargs
2026-05-27 16:22:23 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:24 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
  ✓  PASS  T2.3  self.device is set correctly
2026-05-27 16:22:24 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:24 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
  ✓  PASS  T2.4  Accumulator attributes initialised to zero/empty


### Test Group 3 — `update()` Method

The `update()` method must:
- Accept `(images, prompts, _metadata=None)` without raising
- Increment `_total_count` by `len(images)` per call
- Increment `_evaluated_count` only for successfully scored images
- Append one entry to `_per_image_scores` per image (float or None)
- Accumulate correctly across multiple calls
- Never store raw PIL images (memory safety)

In [6]:
print("=" * 60)
print("Test Group 3: update() Method")
print("=" * 60)

# --------------------------------------------------------------------------
# T3.1  update() increments _total_count  (per_image) /
#        populates _gen_activations        (distribution)
# --------------------------------------------------------------------------
def t3_1_total_count_increments():
    metric = _make_metric()
    imgs    = [_dummy_image(), _dummy_image()]
    prompts = ["a cat", "a dog"]
    metric.update(imgs, prompts)
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        assert metric._total_count == 2, (
            f"After update() with 2 images, _total_count should be 2, got {metric._total_count}. "
            "Increment self._total_count += 1 for every image (even failed ones)."
        )
    else:
        assert len(metric._gen_activations) > 0, (
            "After update() with 2 images, _gen_activations should be non-empty. "
            "Append extracted feature arrays inside update()."
        )

_run("T3.1  update() increments _total_count", t3_1_total_count_increments)

# --------------------------------------------------------------------------
# T3.2  update() appends to _per_image_scores (per_image) /
#        appends to _gen_activations          (distribution)
# --------------------------------------------------------------------------
def t3_2_per_image_scores_appended():
    metric = _make_metric()
    imgs    = [_dummy_image(), _dummy_image(), _dummy_image()]
    prompts = ["p1", "p2", "p3"]
    metric.update(imgs, prompts)
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        assert len(metric._per_image_scores) == 3, (
            f"_per_image_scores must have one entry per image. "
            f"Expected 3, got {len(metric._per_image_scores)}."
        )
    else:
        assert len(metric._gen_activations) >= 1, (
            "After update() with 3 images, _gen_activations must contain at least one "
            "feature-array batch."
        )

_run("T3.2  update() appends one entry per image to _per_image_scores", t3_2_per_image_scores_appended)

# --------------------------------------------------------------------------
# T3.3  Scores in _per_image_scores are float or None (per_image) /
#        _gen_activations entries are numpy arrays     (distribution)
# --------------------------------------------------------------------------
def t3_3_scores_are_float_or_none():
    import numpy as np
    metric = _make_metric()
    metric.update([_dummy_image()], ["a prompt"])
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        for i, score in enumerate(metric._per_image_scores):
            assert score is None or isinstance(score, (int, float)), (
                f"_per_image_scores[{i}] must be float or None, got {type(score).__name__}: {score!r}. "
                "Do not store PIL Image objects in _per_image_scores."
            )
    else:
        for arr in metric._gen_activations:
            assert isinstance(arr, np.ndarray), (
                f"_gen_activations must contain numpy arrays (feature vectors), "
                f"got {type(arr).__name__}."
            )

_run("T3.3  _per_image_scores entries are float or None", t3_3_scores_are_float_or_none)

# --------------------------------------------------------------------------
# T3.4  Accumulates correctly across multiple calls
# --------------------------------------------------------------------------
def t3_4_multi_call_accumulation():
    metric = _make_metric()
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        metric.update([_dummy_image()], ["first"])
        count_after_1 = metric._total_count
        metric.update([_dummy_image(), _dummy_image()], ["second", "third"])
        count_after_2 = metric._total_count
        assert count_after_1 == 1, (
            f"After first update (1 image), _total_count should be 1, got {count_after_1}"
        )
        assert count_after_2 == 3, (
            f"After second update (2 images), _total_count should be 3, got {count_after_2}. "
            "Make sure you accumulate (+=) rather than overwrite (=)."
        )
    else:
        metric.update([_dummy_image()], ["first"])
        batches_after_1 = len(metric._gen_activations)
        metric.update([_dummy_image(), _dummy_image()], ["second", "third"])
        batches_after_2 = len(metric._gen_activations)
        assert batches_after_2 > batches_after_1, (
            f"_gen_activations must grow across update() calls. "
            f"Before: {batches_after_1} batches, after: {batches_after_2} batches. "
            "Make sure update() appends rather than overwrites."
        )

_run("T3.4  Accumulates _total_count correctly across multiple calls", t3_4_multi_call_accumulation)

# --------------------------------------------------------------------------
# T3.5  update() accepts _metadata=None without raising
# --------------------------------------------------------------------------
def t3_5_metadata_none_accepted():
    metric = _make_metric()
    metric.update([_dummy_image()], ["a prompt"], _metadata=None)  # must not raise

_run("T3.5  update() accepts _metadata=None without raising", t3_5_metadata_none_accepted)

# --------------------------------------------------------------------------
# T3.6  update() accepts _metadata as a dict without raising
# --------------------------------------------------------------------------
def t3_6_metadata_dict_accepted():
    metric = _make_metric()
    meta = {"category": "nudity", "source": "i2p"}
    metric.update([_dummy_image()], ["a prompt"], _metadata=meta)  # must not raise

_run("T3.6  update() accepts _metadata as a dict without raising", t3_6_metadata_dict_accepted)

# --------------------------------------------------------------------------
# T3.7  update() does not store raw PIL Image objects
# --------------------------------------------------------------------------
def t3_7_no_raw_image_storage():
    metric = _make_metric()
    img = _dummy_image()
    metric.update([img], ["a prompt"])
    # Walk all instance attributes and check none hold a PIL Image
    for attr, val in vars(metric).items():
        if isinstance(val, Image.Image):
            raise AssertionError(
                f"self.{attr} stores a raw PIL Image after update(). "
                "Compute scores inside update() and discard the image — "
                "do not accumulate raw images as it exhausts RAM."
            )
        if isinstance(val, list):
            for item in val:
                if isinstance(item, Image.Image):
                    raise AssertionError(
                        f"self.{attr} contains a raw PIL Image. "
                        "Score inside update() and store only the float."
                    )

_run("T3.7  update() does not store raw PIL Image objects", t3_7_no_raw_image_storage)


Test Group 3: update() Method
2026-05-27 16:22:24 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:25 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
  ✓  PASS  T3.1  update() increments _total_count
2026-05-27 16:22:25 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:25 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
  ✓  PASS  T3.2  update() appends one entry per image to _per_image_scores
2026-05-27 16:22:25 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:25 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
  ✓  PASS  T3.3  _per_image_scores entries are float or None
2026-05-27 16:22:26 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:26 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
  ✓  PASS  T3.4  Accumulates _total_count correctly across multiple 

### Test Group 4 — `compute()` Method

The `compute()` method must:
- Return a `MetricResult` instance
- Return `MetricResult(value=0.0)` (or similar) when no images were evaluated
- Return correct average when some images failed to score
- Include `evaluated_count`, `total_count`, `per_image_scores`, and `config` in `details`

In [7]:
print("=" * 60)
print("Test Group 4: compute() Method")
print("=" * 60)

from eval_unlearn.types import MetricResult
import numpy as np

def _metric_with_state(total_score, evaluated_count, total_count, per_image_scores):
    """Return a metric with pre-populated accumulator state.

    per_image    → injects the four standard score accumulators directly.
    distribution → injects small synthetic activation arrays (dim=15, full-rank
                   covariance) so compute() can run without calling load_dataset().
                   evaluated_count == 0 triggers the 'no generated images' path.
    """
    metric = _make_metric()
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        metric._total_score       = total_score
        metric._evaluated_count   = evaluated_count
        metric._total_count       = total_count
        metric._per_image_scores  = per_image_scores
    else:
        # Use a small feature dimension so covariance and matrix square-root are fast.
        _DIM = 15
        rng  = np.random.default_rng(42)
        n_real = max(total_count, _DIM + 5)    # enough samples for full-rank sigma_real
        metric._real_activations = rng.standard_normal((n_real, _DIM)).astype(np.float32)
        metric._real_count       = n_real
        if evaluated_count == 0:
            metric._gen_activations = []        # triggers the "no generated images" branch
        else:
            n_gen = max(evaluated_count, _DIM + 2)  # full-rank sigma_gen
            metric._gen_activations = [rng.standard_normal((n_gen, _DIM)).astype(np.float32)]
    return metric

# --------------------------------------------------------------------------
# T4.1  compute() returns a MetricResult instance
# --------------------------------------------------------------------------
def t4_1_returns_metric_result():
    metric = _metric_with_state(1.5, 2, 2, [0.7, 0.8])
    result = metric.compute()
    assert isinstance(result, MetricResult), (
        f"compute() must return eval_unlearn.types.MetricResult, got {type(result).__name__}. "
        "Import MetricResult from ...types and return it."
    )

_run("T4.1  compute() returns MetricResult", t4_1_returns_metric_result)

# --------------------------------------------------------------------------
# T4.2  compute() value is a float
# --------------------------------------------------------------------------
def t4_2_value_is_float():
    metric = _metric_with_state(2.4, 3, 3, [0.8, 0.8, 0.8])
    result = metric.compute()
    assert isinstance(result.value, (int, float)), (
        f"MetricResult.value must be a float, got {type(result.value).__name__}"
    )

_run("T4.2  MetricResult.value is a float", t4_2_value_is_float)

# --------------------------------------------------------------------------
# T4.3  compute() with no images returns a sentinel value and 'error' in details
#
# per_image    → must return value=0.0
# distribution → may return value=inf (no generated images to compare against)
# --------------------------------------------------------------------------
def t4_3_empty_returns_zero():
    import math
    metric = _metric_with_state(0.0, 0, 0, [])
    result = metric.compute()
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        assert result.value == 0.0, (
            f"compute() with zero images must return value=0.0, got {result.value}."
        )
    else:
        assert result.value == 0.0 or not math.isfinite(result.value), (
            f"compute() with no generated images must return 0.0 or a non-finite sentinel "
            f"(e.g. inf), got {result.value}."
        )
    assert "error" in result.details, (
        "compute() with no images must include 'error' in result.details "
        "so callers can distinguish 'no data' from 'score is actually 0'."
    )

_run("T4.3  compute() with no images → value=0.0 and 'error' in details", t4_3_empty_returns_zero)

# --------------------------------------------------------------------------
# T4.4  compute() value is correct
#
# per_image    → checks the exact average: _total_score / _evaluated_count
# distribution → checks the result is a finite, non-negative distance value
# --------------------------------------------------------------------------
def t4_4_average_correct():
    import math
    metric = _metric_with_state(2.1, 3, 3, [0.7, 0.7, 0.7])
    result = metric.compute()
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        expected = 2.1 / 3
        assert math.isclose(result.value, expected, rel_tol=1e-5), (
            f"Expected average {expected:.6f}, got {result.value:.6f}. "
            "Compute: avg = _total_score / _evaluated_count."
        )
    else:
        # Distribution metrics return a distance score, not a simple average.
        assert isinstance(result.value, (int, float)), (
            f"MetricResult.value must be a number, got {type(result.value).__name__}"
        )
        assert result.value >= 0, (
            f"Distribution metric value must be >= 0, got {result.value}."
        )

_run("T4.4  compute() average is correct", t4_4_average_correct)

# --------------------------------------------------------------------------
# T4.5  compute() handles partial success (some images scored None)
#
# per_image    → averages over successfully scored images only
# distribution → returns a valid numeric distance regardless
# --------------------------------------------------------------------------
def t4_5_partial_success():
    import math
    metric = _metric_with_state(1.6, 2, 3, [0.8, None, 0.8])
    result = metric.compute()
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        # 2 scored, 1 failed → average over evaluated images only
        expected = 1.6 / 2
        assert math.isclose(result.value, expected, rel_tol=1e-5), (
            f"Expected {expected:.4f} (average over evaluated images only), got {result.value:.4f}."
        )
    else:
        # Distribution metrics operate on feature distributions, not per-image scores.
        assert isinstance(result.value, (int, float)), (
            f"MetricResult.value must be a number, got {type(result.value).__name__}"
        )

_run("T4.5  compute() handles partial success (some None scores)", t4_5_partial_success)

# --------------------------------------------------------------------------
# T4.6  compute() details contains required keys
#
# per_image    → {"evaluated_count", "total_count", "per_image_scores", "config"}
# distribution → {"total_generated", "total_real", "config"}
# --------------------------------------------------------------------------
def t4_6_details_keys():
    metric = _metric_with_state(0.9, 1, 1, [0.9])
    result = metric.compute()
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        required_keys = {"evaluated_count", "total_count", "per_image_scores", "config"}
    else:
        # Distribution metrics expose dataset-size counters rather than per-image scores.
        required_keys = {"total_generated", "total_real", "config"}
    missing = required_keys - set(result.details.keys())
    assert not missing, (
        f"MetricResult.details is missing required keys: {sorted(missing)}. "
        "Add them to the dict passed to MetricResult(details=...)."
    )

_run("T4.6  details contains: evaluated_count, total_count, per_image_scores, config", t4_6_details_keys)

# --------------------------------------------------------------------------
# T4.7  compute() details['config'] is a dict
# --------------------------------------------------------------------------
def t4_7_config_in_details():
    metric = _metric_with_state(0.5, 1, 1, [0.5])
    result = metric.compute()
    cfg_val = result.details.get("config")
    assert isinstance(cfg_val, dict), (
        f"details['config'] must be a dict (from config.to_dict()), got {type(cfg_val).__name__}."
    )

_run("T4.7  details['config'] is a dict", t4_7_config_in_details)


Test Group 4: compute() Method
2026-05-27 16:22:27 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:27 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
2026-05-27 16:22:27 - eval_unlearn.metrics.fid.metric - INFO - Computing FID: 20 real vs 17 generated images...
2026-05-27 16:22:27 - eval_unlearn.metrics.fid.metric - INFO - FID Score: 6.9903
  ✓  PASS  T4.1  compute() returns MetricResult
2026-05-27 16:22:27 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:28 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
2026-05-27 16:22:28 - eval_unlearn.metrics.fid.metric - INFO - Computing FID: 20 real vs 17 generated images...
2026-05-27 16:22:28 - eval_unlearn.metrics.fid.metric - INFO - FID Score: 6.9903
  ✓  PASS  T4.2  MetricResult.value is a float
2026-05-27 16:22:28 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:28 - eval_unlearn.me

### Test Group 5 — Registry & Entry Point

The metric must be discoverable by the eval-unlearn plugin system after
`pip install -e .`.

In [8]:
print("=" * 60)
print("Test Group 5: Registry & Entry Point")
print("=" * 60)

# --------------------------------------------------------------------------
# T5.1  @register_metric decorator registers in the local registry
# --------------------------------------------------------------------------
def t5_1_registered_in_local_registry():
    with _patch_external():
        _reload_metric_module()   # importing triggers @register_metric

    from eval_unlearn.registry import get_metric
    cls = get_metric(METRIC_NAME)
    assert cls is not None, (
        f"get_metric('{METRIC_NAME}') returned None. "
        f"Make sure your metric class uses @register_metric('{METRIC_NAME}')."
    )

_run("T5.1  @register_metric registers in local registry", t5_1_registered_in_local_registry)

# --------------------------------------------------------------------------
# T5.2  Entry point declared in pyproject.toml
# --------------------------------------------------------------------------
def t5_2_entry_point_in_pyproject():
    import pathlib, tomllib
    root = pathlib.Path("../../../")
    pyproject_path = root / "pyproject.toml"
    if not pyproject_path.exists():
        pyproject_path = pathlib.Path("../../../../pyproject.toml")

    assert pyproject_path.exists(), (
        f"pyproject.toml not found at {pyproject_path.resolve()}."
    )
    with open(pyproject_path, "rb") as f:
        data = tomllib.load(f)

    eps = data.get("project", {}).get("entry-points", {}).get("eval_unlearn.metrics", {})
    assert METRIC_NAME in eps, (
        f"'{METRIC_NAME}' not found under [project.entry-points.\"eval_unlearn.metrics\"] "
        f"in pyproject.toml. Found: {list(eps.keys())}"
    )

_run("T5.2  Entry point declared in pyproject.toml", t5_2_entry_point_in_pyproject)

# --------------------------------------------------------------------------
# T5.3  Entry point dotted path resolves to the correct class
# --------------------------------------------------------------------------
def t5_3_entry_point_path_resolves():
    import pathlib, tomllib
    root = pathlib.Path("../../../")
    pyproject_path = root / "pyproject.toml"
    if not pyproject_path.exists():
        pyproject_path = pathlib.Path("../../../../pyproject.toml")

    with open(pyproject_path, "rb") as f:
        data = tomllib.load(f)

    eps      = data["project"]["entry-points"]["eval_unlearn.metrics"]
    spec     = eps[METRIC_NAME]
    mod_path, cls_name = spec.rsplit(":", 1)

    with _patch_external():
        mod = importlib.import_module(mod_path)

    assert hasattr(mod, cls_name), (
        f"Entry point '{spec}' specifies class '{cls_name}' but it was not found "
        f"in module '{mod_path}'."
    )

_run("T5.3  Entry point path resolves to the metric class", t5_3_entry_point_path_resolves)

Test Group 5: Registry & Entry Point
  ✓  PASS  T5.1  @register_metric registers in local registry
  ✓  PASS  T5.2  Entry point declared in pyproject.toml
  ✓  PASS  T5.3  Entry point path resolves to the metric class


### Test Group 6 — Full Workflow (mocked)

Simulates the complete lifecycle the `SingleBenchmarkRunner` runs: `update()`
called in a loop, followed by `compute()`. The result must be a valid
`MetricResult` with a non-negative value.

In [9]:
print("=" * 60)
print("Test Group 6: Full Workflow")
print("=" * 60)

from eval_unlearn.types import MetricResult
import numpy as np

def _prep_distribution_metric(metric, n_real=20):
    """Inject dim=1 synthetic activations and a fast feature-extraction stub.

    Using a 1-dimensional feature space means:
    - covariance matrices are 1×1 scalars → sqrtm is trivial
    - full-rank sigma is achieved with just 2+ generated images
    This lets the workflow tests complete in milliseconds on CPU.
    """
    _DIM = 1
    rng = np.random.default_rng(0)
    metric._real_activations = rng.standard_normal((n_real, _DIM)).astype(np.float32)
    metric._real_count = n_real
    _call_n = [0]
    def _stub_extract(imgs):
        _call_n[0] += 1
        return np.random.default_rng(_call_n[0]).standard_normal(
            (len(imgs), _DIM)).astype(np.float32)
    metric._extract_features = _stub_extract

# --------------------------------------------------------------------------
# T6.1  update × N followed by compute() returns a valid MetricResult
# --------------------------------------------------------------------------
def t6_1_full_workflow():
    metric  = _make_metric()
    prompts = ["a cat sitting on a mat", "a dog running in the park", "a red apple"]
    images  = [_dummy_image() for _ in prompts]

    if globals().get("METRIC_PATTERN", "per_image") != "per_image":
        _prep_distribution_metric(metric)

    # Simulate 3 batches of 1 image each (typical runner pattern)
    for img, prompt in zip(images, prompts):
        metric.update([img], [prompt])

    result = metric.compute()

    assert isinstance(result, MetricResult), (
        f"Expected MetricResult, got {type(result).__name__}"
    )
    assert result.value >= 0, (
        f"MetricResult.value must be >= 0, got {result.value}."
    )
    if globals().get("METRIC_PATTERN", "per_image") == "per_image":
        assert result.details.get("total_count") == 3, (
            f"Expected total_count=3, got {result.details.get('total_count')}"
        )
    else:
        assert result.details.get("total_generated") == 3, (
            f"Expected total_generated=3, got {result.details.get('total_generated')}"
        )

_run("T6.1  Full workflow: update × 3 → compute() → valid MetricResult", t6_1_full_workflow)

# --------------------------------------------------------------------------
# T6.2  Calling compute() twice gives the same result (idempotent)
# --------------------------------------------------------------------------
def t6_2_compute_idempotent():
    import math
    metric = _make_metric()

    if globals().get("METRIC_PATTERN", "per_image") != "per_image":
        _prep_distribution_metric(metric)

    metric.update([_dummy_image(), _dummy_image()], ["p1", "p2"])

    result1 = metric.compute()
    result2 = metric.compute()

    assert math.isclose(result1.value, result2.value, rel_tol=1e-9), (
        f"compute() is not idempotent: first call returned {result1.value}, "
        f"second call returned {result2.value}. "
        "compute() must read-only — do not modify accumulator state."
    )

_run("T6.2  compute() is idempotent (safe to call multiple times)", t6_2_compute_idempotent)

# --------------------------------------------------------------------------
# T6.3  MetricResult.name is a non-empty string
# --------------------------------------------------------------------------
def t6_3_result_name_nonempty():
    metric = _make_metric()

    if globals().get("METRIC_PATTERN", "per_image") != "per_image":
        _prep_distribution_metric(metric)

    metric.update([_dummy_image()], ["a prompt"])
    result = metric.compute()
    assert isinstance(result.name, str) and result.name, (
        f"MetricResult.name must be a non-empty string, got {result.name!r}."
    )

_run("T6.3  MetricResult.name is a non-empty string", t6_3_result_name_nonempty)


Test Group 6: Full Workflow
2026-05-27 16:22:30 - eval_unlearn.metrics.fid.metric - INFO - Loading InceptionV3 on cpu...
2026-05-27 16:22:31 - eval_unlearn.metrics.fid.metric - INFO - FIDMetric initialized.
2026-05-27 16:22:31 - eval_unlearn.metrics.fid.metric - INFO - Computing FID: 20 real vs 3 generated images...
2026-05-27 16:22:31 - eval_unlearn.metrics.fid.metric - ERROR - FID computation failed.
Traceback (most recent call last):
  File "/vol/bitbucket/m24/eval-unlearn-testing/Packages/eval-unlearn/src/eval_unlearn/metrics/fid/metric.py", line 231, in compute
    fid_score = _calculate_fid(mu_real, sigma_real, mu_gen, sigma_gen)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/vol/bitbucket/m24/eval-unlearn-testing/Packages/eval-unlearn/src/eval_unlearn/metrics/fid/metric.py", line 40, in _calculate_fid
    covmean, _ = linalg.sqrtm(sigma1.dot(sigma2), disp=False)
    ^^^^^^^^^^
ValueError: not enough values to unpack (expected 2, got 1)
  ✗  FAIL 

### Test Group 7 — Integration with SingleBenchmarkRunner

Verifies that the metric can be driven end-to-end through the standard runner
with a lightweight technique (`free_run` mocked).

In [10]:
print("=" * 60)
print("Test Group 7: SingleBenchmarkRunner Integration (mocked)")
print("=" * 60)

# --------------------------------------------------------------------------
# T7.1  Runner accepts the metric name without raising
# --------------------------------------------------------------------------
def t7_1_runner_accepts_metric():
    from eval_unlearn.runners import SingleBenchmarkRunner

    # Ensure metric is registered
    with _patch_external():
        _reload_metric_module()

    # We mock the technique side so only the metric path is exercised
    mock_diffusion = MagicMock()
    mock_diffusion.from_pretrained.return_value = MagicMock()

    with patch.dict("sys.modules", {"diffusers": MagicMock(DiffusionPipeline=mock_diffusion)}):
        runner = SingleBenchmarkRunner(
            technique_name   = "free_run",
            metric_name      = METRIC_NAME,
            technique_config = {"model_id": "fake/model", "device": "cpu"},
            metric_config    = {**REQUIRED_CONFIG, "limit": 1},
            output_dir       = f"/tmp/eval_unlearn_metric_test_{METRIC_NAME}",
            seed             = 0,
        )
    assert runner is not None

_run("T7.1  SingleBenchmarkRunner accepts metric name", t7_1_runner_accepts_metric)

# --------------------------------------------------------------------------
# T7.2  runner.run() returns a report with metric_result
# --------------------------------------------------------------------------
def t7_2_runner_run_returns_report():
    from eval_unlearn.runners import SingleBenchmarkRunner
    from eval_unlearn.types import MetricResult, Dataset
    from torch.utils.data import DataLoader

    with _patch_external():
        _reload_metric_module()

    mock_diffusion = MagicMock()
    mock_diffusion.from_pretrained.return_value = MagicMock()

    with patch.dict("sys.modules", {"diffusers": MagicMock(DiffusionPipeline=mock_diffusion)}):
        runner = SingleBenchmarkRunner(
            technique_name   = "free_run",
            metric_name      = METRIC_NAME,
            technique_config = {"model_id": "fake/model", "device": "cpu"},
            metric_config    = {**REQUIRED_CONFIG, "limit": 1},
            output_dir       = f"/tmp/eval_unlearn_metric_test_{METRIC_NAME}",
            seed             = 0,
        )

    # The runner instantiates technique and metric inside run(), not at __init__ time,
    # so we replace the factory callables (stored as runner.metric_factory /
    # runner.technique_factory) with mocks that return lightweight stand-ins.
    dummy_batch  = Dataset(prompts=["a cat"], metadata={"source": "test", "total_loaded": 1})
    dummy_loader = DataLoader([0], collate_fn=lambda _: dummy_batch)

    mock_metric_inst = MagicMock()
    mock_metric_inst.load_dataset.return_value = dummy_loader
    mock_metric_inst.compute.return_value = MetricResult(
        name="TestMetric", value=0.0, details={}
    )

    mock_technique_inst = MagicMock()
    mock_technique_inst.generate.return_value = [_dummy_image()]

    runner.metric_factory    = MagicMock(return_value=mock_metric_inst)
    runner.technique_factory = MagicMock(return_value=mock_technique_inst)

    report = runner.run()

    assert isinstance(report, dict), f"run() must return a dict, got {type(report)}"
    assert "metric_result" in report, (
        f"report missing 'metric_result'. Keys: {list(report.keys())}"
    )

_run("T7.2  runner.run() returns a report dict with metric_result", t7_2_runner_run_returns_report)


Test Group 7: SingleBenchmarkRunner Integration (mocked)
  ✓  PASS  T7.1  SingleBenchmarkRunner accepts metric name
2026-05-27 16:22:33 - eval_unlearn.runners.single_benchmark_runner - INFO - Starting Benchmark Run...
2026-05-27 16:22:33 - eval_unlearn.runners.single_benchmark_runner - INFO - Run ID: 9b2f9b90
2026-05-27 16:22:33 - eval_unlearn.runners.core.base_runner - INFO - Initializing metric...
2026-05-27 16:22:33 - eval_unlearn.runners.core.base_runner - INFO - Loading dataset...
2026-05-27 16:22:33 - eval_unlearn.runners.core.base_runner - INFO - Initializing technique...
2026-05-27 16:22:33 - eval_unlearn.runners.core.base_runner - INFO - Generating images and computing metrics...
2026-05-27 16:22:33 - eval_unlearn.artifacts.writer - INFO - Saving 1 images to /tmp/eval_unlearn_metric_test_fid/free_run_fid_9b2f9b90/images...
2026-05-27 16:22:33 - eval_unlearn.artifacts.writer - INFO - Skipping report save (not provided)
2026-05-27 16:22:33 - eval_unlearn.runners.single_benchmark

---
## Summary

In [11]:
print("=" * 60)
print(f"Contribution checklist for metric: {METRIC_NAME!r}")
print("=" * 60)

checklist = [
    "src/eval_unlearn/metrics/{name}/__init__.py exists",
    "src/eval_unlearn/metrics/{name}/config.py defines {cls}Config(BaseConfig)",
    "src/eval_unlearn/metrics/{name}/metric.py defines {cls} with @register_metric",
    "Config is frozen (@dataclass(frozen=True))",
    "Config.from_dict / to_dict round-trips cleanly",
    "__init__(self, **kwargs): loads model, sets self.device and accumulators",
    "load_dataset(self) -> |DataLoader: resets accumulators, returns DataLoader",
    "update(self, images, prompts, _metadata=None): increments _total_count per image",
    "update() does NOT store raw PIL Images in instance state",
    "compute() -> MetricResult: divides accumulators, includes required details keys",
    "compute() returns value=0.0 and 'error' in details when _total_count == 0",
    "Entry point added to pyproject.toml under eval_unlearn.metrics",
    "pip install -e . completed after editing pyproject.toml",
    "Optional: metric documented in src/eval_unlearn/metrics/_base_models.py",
]

for item in checklist:
    line = item.format(name=METRIC_NAME, cls=METRIC_CLASS)
    print(f"  [ ]  {line}")

print()
print("Once all tests above show PASS and this checklist is complete,")
print("open a pull request against the main branch.")
print()
print("To run the full automated test suite:")
print("  cd /path/to/eval-unlearn && pytest -m 'not integration' -v")

Contribution checklist for metric: 'fid'
  [ ]  src/eval_unlearn/metrics/fid/__init__.py exists
  [ ]  src/eval_unlearn/metrics/fid/config.py defines FIDMetricConfig(BaseConfig)
  [ ]  src/eval_unlearn/metrics/fid/metric.py defines FIDMetric with @register_metric
  [ ]  Config is frozen (@dataclass(frozen=True))
  [ ]  Config.from_dict / to_dict round-trips cleanly
  [ ]  __init__(self, **kwargs): loads model, sets self.device and accumulators
  [ ]  load_dataset(self) -> |DataLoader: resets accumulators, returns DataLoader
  [ ]  update(self, images, prompts, _metadata=None): increments _total_count per image
  [ ]  update() does NOT store raw PIL Images in instance state
  [ ]  compute() -> MetricResult: divides accumulators, includes required details keys
  [ ]  compute() returns value=0.0 and 'error' in details when _total_count == 0
  [ ]  Entry point added to pyproject.toml under eval_unlearn.metrics
  [ ]  pip install -e . completed after editing pyproject.toml
  [ ]  Optional: 